In [1]:
import pandas as pd
import numpy as np
import re
from google.colab import drive
from sklearn.model_selection import GroupShuffleSplit

# Mount Drive and load dataset
drive.mount('/content/drive')
FILE_PATH = '/content/drive/MyDrive/Finlens/data/processed/Finlens_dataset.csv'

# Load the dataset
df = pd.read_csv(FILE_PATH, engine='python', quotechar='"', on_bad_lines='skip')

print(f"Initial Dataset Shape: {df.shape}")
print(f"Unique Providers Found: {df['provider_name'].nunique()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Initial Dataset Shape: (9467, 12)
Unique Providers Found: 56


In [2]:
def clean_legal_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Remove HTML tags that might have survived scraping
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Replace non-breaking spaces and newline variations with a standard space
    text = re.sub(r'[\r\n\xa0]+', ' ', text)
    
    # Remove excessive whitespace
    text = re.sub(r'\s{2,}', ' ', text)
    
    # Strip leading/trailing spaces
    return text.strip()

# Apply cleaning, ensuring we don't overwrite the original raw_text
df['cleaned_clause_text'] = df['raw_text'].apply(clean_legal_text)

print("Sample comparison (Before vs After):")
print("Original:", df['raw_text'].iloc[0][:100], "...")
print("Cleaned: ", df['cleaned_clause_text'].iloc[0][:100], "...")

Sample comparison (Before vs After):
Original: Tala Loan App Kenya | Line of Credit Agreement Terms
Skip to content
Line of Credit Agreement
On our ...
Cleaned:  Tala Loan App Kenya | Line of Credit Agreement Terms Skip to content Line of Credit Agreement On our ...


In [3]:
initial_rows = len(df)

# Drop exact duplicates within the same provider
df_dedup = df.drop_duplicates(subset=['provider_name', 'cleaned_clause_text'], keep='first')

dedup_rows = len(df_dedup)
print(f"Removed {initial_rows - dedup_rows} intra-provider duplicates.")
print(f"Remaining clauses for splitting: {dedup_rows}")

Removed 52 intra-provider duplicates.
Remaining clauses for splitting: 9415


In [6]:
compliant_providers = sorted(
    df_dedup.loc[
        df_dedup['violation_category'].eq('COMPLIANT'),
        'provider_name'
    ].unique()
)

print("Providers with compliant clauses:")
print(*compliant_providers, sep="\n")

Providers with compliant clauses:
Advancepoa
Asante
Asapcredit
Asapkash
Autocheck
Bayes
Berry
Betapesa
Boreleza
Branch
Chapaa Loan
Chapeo
Choice Sasa
Coop Bank
FairKash+
Flashpesa
Ganji Pesa
Helapesa
Helasasa
Jumo
Kashbean
Kechita Capital
Kikwetu Credit
Kopokopo
Koro
Lendplus
Letsgo
Littlepesa
Loanplus
Lockbx
Loop
Malicash
Meta Loan
Mkopa
Numida
Pato Capital
Pawacash
Pesain
Pezesha
Premier Credit
Raha Pesa
Spectrum Credit
Tala
Tanir Credit
Tazu Credit
Tick Credit
Timiza
Tuma Loan
Tumacash
ZK Pesa
Zanifu
Zash Loan
Zenka
Zuri Cash
iLoan
iPesa


In [11]:
# 4. Manual Stratified Provider Routing (Addressing severe class imbalance)
# Route specific providers to ensure all rare classes are represented in evaluation
# Drop the unlearnable minority class before splitting
df_dedup = df_dedup[df_dedup['violation_category'] != 'OTHER_REGULATORY_CONCERN'].copy()

# 4. Manual Stratified Provider Routing (Optimized for FinLens Class Balance)
test_providers = ['Kechita Capital', 'Coop Bank', 'Meta Loan', 'Koro', 'Tala'] 
val_providers = ['Zenka', 'Choice Sasa', 'Zanifu', 'Kopokopo', 'Helasasa']

# Assign remaining providers to Training
all_providers = df_dedup['provider_name'].unique()
train_providers = [p for p in all_providers if p not in test_providers and p not in val_providers]

# 5. Execute the Split
train_df = df_dedup[df_dedup['provider_name'].isin(train_providers)].copy()
val_df = df_dedup[df_dedup['provider_name'].isin(val_providers)].copy()
test_df = df_dedup[df_dedup['provider_name'].isin(test_providers)].copy()

# Print stats to verify the split sizes
print("--- Final Dataset Splits ---")
print(f"Training Set:   {len(train_df)} clauses ({train_df['provider_name'].nunique()} unique lenders)")
print(f"Validation Set: {len(val_df)} clauses ({val_df['provider_name'].nunique()} unique lenders)")
print(f"Test Set:       {len(test_df)} clauses ({test_df['provider_name'].nunique()} unique lenders)")

# Quick sanity check for leakage across all three sets
train_set = set(train_providers)
val_set = set(val_providers)
test_set = set(test_providers)

leakage_train_test = train_set.intersection(test_set)
leakage_train_val = train_set.intersection(val_set)
leakage_val_test = val_set.intersection(test_set)

if not leakage_train_test and not leakage_train_val and not leakage_val_test:
    print("\nSuccess: No data leakage. Train, Val, and Test provider sets are mutually exclusive.")
else:
    print(f"\nWarning! Leakage detected.")
    if leakage_train_test: print(f"Train/Test overlap: {leakage_train_test}")
    if leakage_train_val: print(f"Train/Val overlap: {leakage_train_val}")
    if leakage_val_test: print(f"Val/Test overlap: {leakage_val_test}")

--- Final Dataset Splits ---
Training Set:   7375 clauses (46 unique lenders)
Validation Set: 1197 clauses (5 unique lenders)
Test Set:       841 clauses (5 unique lenders)

Success: No data leakage. Train, Val, and Test provider sets are mutually exclusive.


In [12]:
print("--- Class Distribution ---")
print("\nTraining Set:")
print(train_df['violation_category'].value_counts())

print("\nValidation Set:")
print(val_df['violation_category'].value_counts())

print("\nTest Set:")
print(test_df['violation_category'].value_counts())

--- Class Distribution ---

Training Set:
violation_category
COMPLIANT                         7311
KDPA_EXCESSIVE_DATA_COLLECTION      22
UNDISCLOSED_FEES                    15
AGGRESSIVE_COLLECTION               11
KDPA_CONTACT_ACCESS                  8
KDPA_INADEQUATE_CONSENT              8
Name: count, dtype: int64

Validation Set:
violation_category
COMPLIANT                         1184
UNDISCLOSED_FEES                     4
KDPA_EXCESSIVE_DATA_COLLECTION       3
AGGRESSIVE_COLLECTION                3
KDPA_CONTACT_ACCESS                  2
KDPA_INADEQUATE_CONSENT              1
Name: count, dtype: int64

Test Set:
violation_category
COMPLIANT                         832
KDPA_CONTACT_ACCESS                 2
KDPA_EXCESSIVE_DATA_COLLECTION      2
UNDISCLOSED_FEES                    2
AGGRESSIVE_COLLECTION               2
KDPA_INADEQUATE_CONSENT             1
Name: count, dtype: int64


In [13]:
import os

# Define export directory
EXPORT_DIR = '/content/drive/MyDrive/Finlens/data/splits/'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save to CSV
train_df.to_csv(os.path.join(EXPORT_DIR, 'train_split.csv'), index=False)
val_df.to_csv(os.path.join(EXPORT_DIR, 'val_split.csv'), index=False)
test_df.to_csv(os.path.join(EXPORT_DIR, 'test_split.csv'), index=False)

print(f"Preprocessing complete. Splits successfully saved to {EXPORT_DIR}")

Preprocessing complete. Splits successfully saved to /content/drive/MyDrive/Finlens/data/splits/


In [14]:
LOCAL_EXPORT_DIR = os.path.join(os.getcwd(), "data", "splits")
os.makedirs(LOCAL_EXPORT_DIR, exist_ok=True)

train_df.to_csv(os.path.join(LOCAL_EXPORT_DIR, "train_split.csv"), index=False)
val_df.to_csv(os.path.join(LOCAL_EXPORT_DIR, "val_split.csv"), index=False)
test_df.to_csv(os.path.join(LOCAL_EXPORT_DIR, "test_split.csv"), index=False)

print(f"Splits saved locally to: {LOCAL_EXPORT_DIR}")

Splits saved locally to: /content/data/splits
